In [1]:
def get_effective_unitary(gate, ancilla_indices, ancilla_state=0):
    """
    Extract the effective unitary on logical qubits when ancilla qubits
    are fixed to a specific state (default |0⟩).
    
    Args:
        gate: The full gate/circuit
        ancilla_indices: List of qubit indices that are ancillas (0-indexed)
        ancilla_state: The fixed state of ancillas (usually 0)
    
    Returns:
        The effective unitary on the logical qubits
    """
    full_unitary = Operator(gate).data
    n_qubits = int(np.log2(full_unitary.shape[0]))
    
    # Find which basis states have ancilla in the specified state
    logical_indices = []
    for i in range(2**n_qubits):
        # Check if all ancilla qubits are in the correct state
        ancilla_match = all(
            ((i >> idx) & 1) == ancilla_state 
            for idx in ancilla_indices
        )
        if ancilla_match:
            logical_indices.append(i)
    
    # Extract the submatrix
    effective_U = full_unitary[np.ix_(logical_indices, logical_indices)]
    return effective_U


def verify_effective_gate(gate, target_gate, ancilla_indices, ancilla_state=0):
    """
    Verify that a gate with ancillas implements a target gate effectively.
    """
    effective_U = get_effective_unitary(gate, ancilla_indices, ancilla_state)
    target_U = Operator(target_gate).data
    
    # Check equivalence up to global phase
    if effective_U.shape != target_U.shape:
        return False, "Dimension mismatch"
    
    # Compute U_eff @ U_target^†
    product = effective_U @ target_U.conj().T
    
    # Should be proportional to identity
    phase = product[0, 0]
    if np.abs(phase) < 1e-10:
        return False, "Zero phase - gates not equivalent"
    
    identity_scaled = phase * np.eye(len(product))
    is_equivalent = np.allclose(product, identity_scaled, atol=1e-8)
    
    return is_equivalent, {
        'effective_unitary': effective_U,
        'target_unitary': target_U,
        'global_phase': np.angle(phase),
        'fidelity': np.abs(np.trace(product) / len(product))**2
    }
    
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator

def _unitaries_close_up_to_phase(U, V, atol=1e-8):
    """Check ||e^{-iφ} U - V|| is small for some global phase φ."""
    # Flatten and compute best-fit global phase via inner product
    uv = np.vdot(V.flatten(), U.flatten())  # <V|U>
    if np.abs(uv) < 1e-12:
        # fallback: just do direct allclose (unlikely)
        return np.allclose(U, V, atol=atol)
    phase = np.angle(uv)
    U_phase = np.exp(-1j * phase) * U
    return np.allclose(U_phase, V, atol=atol)


In [2]:
from qiskit.transpiler import TransformationPass, Target
from qiskit.circuit import QuantumCircuit
from qiskit.dagcircuit import DAGCircuit

import numpy as np
from qiskit.circuit import Gate, QuantumCircuit, AncillaQubit
from qiskit import QuantumRegister, AncillaRegister

from qiskit.circuit.library import RZGate, XXPlusYYGate, SGate, SdgGate, CXGate, iSwapGate, GlobalPhaseGate

from qiskit.synthesis import OneQubitEulerDecomposer
from qiskit.quantum_info import Operator
from qiskit.circuit.library import CZGate

from numpy.typing import NDArray
import scipy
from dataclasses import dataclass
from typing import List, Tuple

# from reck_decompose import reck_decomposition, reconstruct_unitary

ComplexMatrix = NDArray[np.complexfloating]
RealVector = NDArray[np.floating]
class SqrtISWAPGate(Gate):
    """Custom √iSWAP gate, built on Qiskit's XXPlusYYGate."""

    def __init__(self):
        super().__init__(name="sqrtiSWAP_ec", num_qubits=2, params=[])

    def _define(self):
        qc = QuantumCircuit(2, name="sqrtiSWAP_ec")
        # qc.append(iSwapGate().power(0.5), [0, 1])
        qc.append(XXPlusYYGate(theta=-np.pi/2), [0, 1])
        self.definition = qc.to_instruction()
    
    def to_matrix(self):
        """Return the matrix representation of the √iSWAP gate."""
        return XXPlusYYGate(theta=-np.pi/2).to_matrix()

class SqrtISWAPdgGate(Gate):
    """Custom √iSWAP^† gate, built on Qiskit's XXPlusYYGate."""

    def __init__(self):
        super().__init__(name="sqrtiSWAPdg_ec", num_qubits=2, params=[])

    def _define(self):
        qc = QuantumCircuit(2, name="sqrtiSWAPdg_ec")
        # qc.append(iSwapGate().power(0.5), [0, 1])
        qc.append(XXPlusYYGate(theta=np.pi/2), [0, 1])
        self.definition = qc.to_instruction()
    
    def to_matrix(self):
        """Return the matrix representation of the √iSWAP gate."""
        return XXPlusYYGate(theta=np.pi/2).to_matrix()


@dataclass
class TwoLevelBlock:
    dim: int              # full dimension of U
    i: int                # first basis index (global)
    j: int                # second basis index (global)
    submatrix: np.ndarray # 2x2 unitary acting on span{|i>, |j>}

@dataclass
class ControlledTwoLevel:
    """A 2-level SU(2) acting on a target pair, conditioned on some control bits."""
    U: np.ndarray              # 2x2 SU(2) matrix
    control_bits: list[int]    # bit positions of controls
    target_bits: list[int]     # two target bit positions [t1, t2]

class CZFromXYGate(Gate):
    """Two-qubit CZ implemented (conceptually) from XY / √iSWAP + Rz.

    For now, the definition is just the standard CZ.
    Replace the circuit in `_define()` with your derived √iSWAP+Rz sequence.
    """

    def __init__(self):
        super().__init__(name="cz_xy", num_qubits=2, params=[])

    def _define(self):
        qr = QuantumRegister(2, "q")
        ar = AncillaRegister(1, "a")  # No ancillas needed here
        qc = QuantumCircuit(qr, ar, name="cz_xy")

        # Example structure (you fill in angles / pattern):
        qc.append(SqrtISWAPGate(), [0, 1])
        qc.append(SqrtISWAPGate(), [0, 1])
        qc.append(SqrtISWAPGate(), [1, 2])
        qc.append(SqrtISWAPGate(), [1, 2])
        
        qc.append(SqrtISWAPdgGate(), [0, 1])
        qc.append(SqrtISWAPdgGate(), [0, 1])
        
        qc.append(SqrtISWAPGate(), [0, 2])
        qc.append(SqrtISWAPGate(), [0, 2])
        
        qc.append(SGate(), [0]),
        qc.append(SdgGate(), [2]),

        self.definition = qc
        qc.initialize('0', ar[0])
        # self._definition = qc.to_gate()

    def inverse(self):
        """CZ is Hermitian and self-inverse."""
        return CZFromXYGate()

class EnergyConservationDecompositionPass(TransformationPass):
    """A custom transpiler pass that decomposes energy-conserving unitaries
    into a basis set of gates that also conserve energy.
    """

    def __init__(self, target: Target):
        super().__init__()
        self.target = target
    
    
    @staticmethod
    def is_energy_conserving(unitary: ComplexMatrix, atol: float = 1e-10) -> bool:
        """Check if a unitary preserves Hamming weight (is energy-conserving).

        An energy-conserving unitary has block-diagonal structure in the
        computational basis, where each block corresponds to states with
        the same Hamming weight.

        Args:
            unitary: The unitary matrix to check.
            atol: Absolute tolerance for numerical comparisons.

        Returns:
            True if the unitary is energy-conserving, False otherwise.
        """
        n = unitary.shape[0]
        n_qubits = int(np.log2(n))

        if 2**n_qubits != n:
            raise ValueError(f"Unitary dimension {n} is not a power of 2")

        # Check that matrix elements between different Hamming weight sectors vanish
        for i in range(n):
            for j in range(n):
                hw_i = bin(i).count('1')
                hw_j = bin(j).count('1')
                if hw_i != hw_j and np.abs(unitary[i, j]) > atol:
                    return False
        return True

    @staticmethod
    def get_hamming_weight_blocks(unitary: ComplexMatrix) -> dict[int, ComplexMatrix]:
        """Extract the block-diagonal structure by Hamming weight sectors.

        Args:
            unitary: Energy-conserving unitary matrix.

        Returns:
            Dictionary mapping Hamming weight to the corresponding unitary block.

        Raises:
            ValueError: If the unitary is not energy-conserving.
        """
        if not EnergyConservationDecompositionPass.is_energy_conserving(unitary):
            raise ValueError("Unitary is not energy-conserving")

        n = unitary.shape[0]
        n_qubits = int(np.log2(n))

        # Group basis states by Hamming weight
        hw_to_indices: dict[int, list[int]] = {}
        for i in range(n):
            hw = bin(i).count('1')
            if hw not in hw_to_indices:
                hw_to_indices[hw] = []
            hw_to_indices[hw].append(i)

        # Extract blocks
        blocks = {}
        for hw, indices in hw_to_indices.items():
            block_size = len(indices)
            block = np.zeros((block_size, block_size), dtype=complex)
            for new_i, old_i in enumerate(indices):
                for new_j, old_j in enumerate(indices):
                    block[new_i, new_j] = unitary[old_i, old_j]
            blocks[hw] = block

        return blocks

    
    
    def _apply_cz_decomposition(self, qc, q0, q1, ancilla):
        """
        Apply CZ on (q0, q1) using ancilla.
        
        The ancilla is assumed to be |0⟩ and should return to |0⟩.
        
        Replace this with your actual derived decomposition!
        """
        # Your √iSWAP decomposition of CZ
        # q0, q1 are the logical qubits
        # ancilla is the helper qubit (starts and ends in |0⟩)
        
        qc.append(SqrtISWAPGate(), [q0, q1])
        qc.append(SqrtISWAPGate(), [q0, q1])
        qc.append(SqrtISWAPGate(), [q1, ancilla])
        qc.append(SqrtISWAPGate(), [q1, ancilla])
        
        qc.append(SqrtISWAPdgGate(), [q0, q1])
        qc.append(SqrtISWAPdgGate(), [q0, q1])
        
        qc.append(SqrtISWAPGate(), [q0, ancilla])
        qc.append(SqrtISWAPGate(), [q0, ancilla])
        
        qc.append(SGate(), [q0])
        qc.append(SdgGate(), [ancilla])
        
    def _apply_swap_decomposition(self, qc, q0, q1, ancilla):
        """
        Apply SWAP on (q0, q1) using ancilla.
        
        The ancilla is assumed to be |0⟩ and should return to |0⟩.
        
        Replace this with your actual derived decomposition!
        """
        # Your √iSWAP decomposition of SWAP
        # q0, q1 are the logical qubits
        # ancilla is the helper qubit (starts and ends in |0⟩)
        
        self._apply_cz_decomposition(qc, q0, q1, ancilla)
        qc.append(SdgGate(), [q0])
        qc.append(SdgGate(), [q1])
        qc.append(SqrtISWAPGate(), [q0, q1])
        qc.append(SqrtISWAPGate(), [q0, q1])
    
    def _apply_2qubit_decomposition(self, qc, instr, qargs):
        q0, q1 = qargs[0], qargs[1]
        U = Operator(instr).data
        
        if not EnergyConservationDecompositionPass.is_energy_conserving(U):
            # apply the gate as is
            print("Non-energy-conserving 2-qubit gate encountered; applying directly.")
            qc.append(instr, [q0, q1])
            return
        
        blocks = EnergyConservationDecompositionPass.get_hamming_weight_blocks(U)
        block_0 = blocks.get(0, np.eye(1))
        theta_0 = np.angle(block_0[0, 0])
        
        block_1 = blocks.get(1, np.eye(2))
        print(f"2-qubit EC block (HW=1): {block_1}")
        theta, phi, lam, global_phase = OneQubitEulerDecomposer('ZYZ').angles_and_phase(Operator(block_1))
        
        print(f"2-qubit EC block angles: θ={theta:.3f}, φ={phi:.3f}, λ={lam:.3f}, global_phase={global_phase:.3f}")
        
        theta_1 = global_phase
        # Apply the decomposition to qc
        qc.rz(-phi/2, q1)
        qc.rz(phi/2, q0)
        qc.append(SqrtISWAPGate(), [q0, q1])
        qc.rz(-theta/2, q1)
        qc.rz(theta/2, q0)
        qc.append(SqrtISWAPdgGate(), [q0, q1])
        qc.rz(-lam/2, q1)
        qc.rz(lam/2, q0)
        
        
        block_1 = blocks.get(2, np.eye(1))
        theta_2 = np.angle(block_1[0, 0])
        # Global phase adjustment (if needed)
        return [theta_0, theta_1, theta_2]
    
    
    def _fix_relative_phases(self, qc, qargs, anc, phases):
        theta_0 = phases[0]
        num_qubits = len(qargs)
        for m, theta in enumerate(phases[1:]):
            delta = theta - theta_0
            
            if np.abs(delta) > 1e-10:
                # Step 1: Pick |b> : any bitstring of weight m+1
                b = None
                for i in range(2**num_qubits):
                    bits = bin(i)[2:].zfill(num_qubits)
                    if bits.count('1') == m + 1:
                        b = bits
                        break

                if b is None:
                    raise ValueError(f"No basis state with Hamming weight {m+1}")

                # Step 2: Construct |b'>
                # flip one '1' → '0'
                b_list = list(b)
                flip_index = None
                for idx in range(num_qubits):
                    if b_list[idx] == '1':
                        b_list[idx] = '0'
                        flip_index = idx
                        break

                b_prime = ''.join(b_list)

                # Step 3: Embed into (n+1)-qubit Hilbert space
                # index(|x> | anc >) = binary string x + anc_bit
                dim = 2**(num_qubits + 1)
                H = np.zeros((dim, dim), dtype=complex)

                # |b> |0>
                idx_b0 = int(b + "0", 2)

                # |b'> |1>
                idx_bp1 = int(b_prime + "1", 2)

                H[idx_b0, idx_b0] = +1   # |b>|0> term
                H[idx_bp1, idx_bp1] = -1 # |b'>|1> term
                
                with np.printoptions(precision=3, suppress=True):
                    print("Applying phase correction:")
                    print(f"Delta for HW {m+1}: {delta:.4f} rad")
                    print("Hamming weight matrix H:")
                    print(H)
                    
                
                #exponentiate
                U_phase = scipy.linalg.expm(1j * delta * H)
                
                # U_phase = np.expm(1j * delta * H)
                
                U_phase_submatrix = U_phase[np.ix_([idx_b0, idx_bp1], [idx_b0, idx_bp1])]
                with np.printoptions(precision=3, suppress=True, linewidth=1000):
                    print("Submatrix to implement:")
                    print(U_phase)
                # qc.append(Operator(U_phase), qargs)   
                # Apply 3 qubit decomposition of U_phase 
                self._apply_2level_3qubit_unitary(qc, U_phase, qargs + [anc])
    
    def _exp_i_alpha_R(self, theta, qc, q0, q1):
        qc.append(SGate(), [q1]),
        qc.append(SqrtISWAPGate(), [q0, q1])
        qc.rz(-theta, q0)
        qc.rz(theta, q1)
        qc.append(SqrtISWAPdgGate(), [q0, q1])
        qc.append(SdgGate(), [q1]),

    def _controlled_exp_i_alpha_R(self,theta, b, qc, q0, q1, control):
            q2 = control
            qc.append(SqrtISWAPGate(), [q1, q2])
            qc.append(SqrtISWAPGate(), [q1, q2])
            
            qc.append(SqrtISWAPdgGate(), [q0, q1])
            qc.append(SqrtISWAPdgGate(), [q0, q1])
        
            qc.append(SqrtISWAPGate(), [q0, q2])
            qc.append(SqrtISWAPGate(), [q0, q2])
            
            qc.append(SGate(), [q0]),
            self._exp_i_alpha_R( (-1)**b * (theta/2), qc, q0, q1)
            qc.append(SdgGate(), [q0]),
            
            qc.append(SqrtISWAPdgGate(), [q0, q2])
            qc.append(SqrtISWAPdgGate(), [q0, q2])
            
            qc.append(SqrtISWAPGate(), [q0, q1])
            qc.append(SqrtISWAPGate(), [q0, q1])
            
            qc.append(SqrtISWAPdgGate(), [q1, q2])
            qc.append(SqrtISWAPdgGate(), [q1, q2])
            
            self._exp_i_alpha_R(theta/2, qc, q0, q1)
            
    def _controlled_exp_i_theta_L(self, theta, b, qc, q0, q1, control):
            q2 = control
            qc.append(SGate(), [q0])
            self._controlled_exp_i_alpha_R( (-1)**b * (theta), b, qc, q0, q1, q2)
            qc.append(SdgGate(), [q0])
            
    
    @staticmethod
    def _get_nontrivial_rows_cols(U, atol=1e-10):
        n = U.shape[0]
        nontrivial_rows = []
        nontrivial_cols = []

        for i in range(n):
            # row i
            nz_row = np.where(np.abs(U[i, :]) > atol)[0]
            # print(f"Row {i} non-zero indices: {nz_row}")
            if not (len(nz_row) == 1 and nz_row[0] == i and np.isclose(np.real(U[i, i]), 1.0, atol=atol)):
                nontrivial_rows.append(i)

            # col i
            nz_col = np.where(np.abs(U[:, i]) > atol)[0]
            # print(f"Col {i} non-zero indices: {nz_col}")
            if not (len(nz_col) == 1 and nz_col[0] == i and np.isclose(np.real(U[i, i]), 1.0, atol=atol)):
                nontrivial_cols.append(i)

        if len(nontrivial_rows) > 2 or len(nontrivial_cols) > 2:
            raise ValueError(f"Unitary is not a valid 2-level unitary, found {len(nontrivial_rows)} non-trivial rows and {len(nontrivial_cols)} non-trivial columns.")

        return nontrivial_rows[0], nontrivial_rows[1]


    
    def _apply_2level_3qubit_unitary(self, qc, U, qargs):
        """
        Apply a 2-level unitary from Reck decomposition to a 3-qubit circuit.
        
        Note: Correctly implements upto global phase of the hamming-weight subspace.
        
        Args:
            qc: QuantumCircuit
            U: Unitary acting on 2 levels
            qargs: [q0, q1, q2] physical qubit indices 
        """
        q0, q1, q2 = qargs # Using q0 as most significant bit
        
        # Check that U is a two level unitary ie. only 2 non-trivial rows/cols
        local_i, local_j = self._get_nontrivial_rows_cols(U)
        
        
        # get submatrix
        submatrix = np.zeros((2, 2), dtype=complex)
        submatrix[0, 0] = U[local_i, local_i]
        submatrix[0, 1] = U[local_i, local_j]
        submatrix[1, 0] = U[local_j, local_i]
        submatrix[1, 1] = U[local_j, local_j]
        
        twolevel_U = TwoLevelUnitary(U.shape[0], local_i, local_j, submatrix)
        
        # XYX decomposition: U = Rx(theta) @ Ry(phi) @ Rx(lam)
        theta, phi, lam, global_phase = OneQubitEulerDecomposer('XYX').angles_and_phase(
            Operator(submatrix)
        )
        
        # Figure out control and target qubits solely from U
        bits_i = bin(local_i)[2:].zfill(3)
        bits_j = bin(local_j)[2:].zfill(3)
        print(f"  Applying TLU: |{bits_i}⟩ ↔ |{bits_j}⟩ on local indices {local_i}, {local_j}")
        
        # # Find the common bit (control qubit) and its value
        control_bit = None
        control_val = None
        target_bits = []
        
        for bit in range(3):
            bit_i = (local_i >> bit) & 1
            bit_j = (local_j >> bit) & 1
            if bit_i == bit_j:
                control_bit = bit
                control_val = bit_i
            else:
                target_bits.append(bit)
        
        print(f"    Control: q{control_bit}={control_val}, Targets: q{target_bits}")
        print(f"    XYX angles: θ={theta:.3f}, φ={phi:.3f}, λ={lam:.3f}")
        
        # Map bit positions to physical qubits
        control = qargs[control_bit]
        target1 = qargs[target_bits[0]]
        target2 = qargs[target_bits[1]]

        # Apply XYX decomposition using controlled rotations
        # U = Rx(theta) @ Ry(phi) @ Rx(lam) in the 2-level subspace
        # Rx uses R rotation, Ry uses L rotation
        
        self._controlled_exp_i_alpha_R(-phi/2, control_val, qc, target1, target2, control)
        self._controlled_exp_i_theta_L(-theta/2, control_val, qc, target1, target2, control)
        self._controlled_exp_i_alpha_R(-lam/2, control_val, qc, target1, target2, control)

    def _decompose_ncontrolled_2level_unitary(self, U, control_bits, target_bits):
        """
        Decompose a 2-level unitary with n control bits and 2 target bits.
        
        Args:
            U: 2 x 2 unitary matrix
            control_bits: list of control bit indices
            target_bits: list of target bit indices (length 2)
        
        Returns:
            List of 2-level unitaries to apply in sequence.
        """
        
        # XYX decomposition: U = Rx(theta) @ Ry(phi) @ Rx(lam)
        theta, phi, lam, global_phase = OneQubitEulerDecomposer('XYX').angles_and_phase(
            Operator(submatrix)
        )
        
        return theta, phi, lam, global_phase
    
    
    @staticmethod
    def _su2_AB_from_U(U: np.ndarray, atol: float = 1e-10) -> tuple[np.ndarray, np.ndarray]:
        """
        Given U ∈ SU(2), find A,B ∈ SU(2) such that A B A† B† = U,
        following the construction in the paper (see Eq. (64)).

        We first factor out global phase to make det(U) = 1, then:
          U = W exp(i θ Z) W†
          A(1) = W exp(i θ Z/2) W†
          B(1) = i W X W†
        """
        if U.shape != (2, 2):
            raise ValueError("U must be 2x2")

        # Remove global phase so that det(U_su2) = 1
        detU = np.linalg.det(U)
        if np.abs(detU) < atol:
            raise ValueError("Determinant of U is ~0, not unitary?")
        U_su2 = U / np.sqrt(detU)

        # Eigen-decomposition: U_su2 = W diag(e^{iφ1}, e^{iφ2}) W†
        evals, evecs = np.linalg.eig(U_su2)
        # Order is arbitrary; just take as given
        φ1, φ2 = np.angle(evals[0]), np.angle(evals[1])

        # For SU(2), φ1 + φ2 ≈ 0 (mod 2π)
        θ = 0.5 * (φ1 - φ2)  # so that eigenvalues are e^{iθ}, e^{-iθ} up to phase

        # Build exp(i θ Z/2) and exp(i θ Z)
        Z = np.diag([1.0, -1.0])
        exp_iθZ_over2 = np.diag(np.exp(1j * θ * np.array([1.0, -1.0]) / 2.0))

        W = evecs  # columns are eigenvectors
        # A(1) = W exp(i θ Z/2) W†
        A1 = W @ exp_iθZ_over2 @ W.conj().T

        # B(1) = i W X W†, where X = [[0,1],[1,0]]
        X = np.array([[0.0, 1.0], [1.0, 0.0]], dtype=complex)
        B1 = 1j * W @ X @ W.conj().T

        # Return A(1), B(1); global phase differences are fine
        return A1, B1
    
        
    @staticmethod
    def extract_two_level_blocks(U: np.ndarray, atol: float = 1e-10) -> List[TwoLevelBlock]:
        """
        Given a unitary U (N x N), find all disjoint 2-level unitaries embedded in it.

        A "2-level" unitary here means:
        - U acts nontrivially only on the span of {|i>, |j>} for some i != j
        - All other basis states are only multiplied by phases (i.e., no off-diagonal
            couplings to them above tolerance).

        If any connected component in the coupling graph has size > 2, we raise,
        because that implies 3+ level mixing.

        Args:
            U: (N x N) unitary matrix.
            atol: numerical tolerance to treat entries as zero.

        Returns:
            List[TwoLevelBlock], one per 2-level subspace.

        Raises:
            ValueError: if U is not square, not unitary (within atol),
                        or contains a >2-dimensional mixing component.
        """
        # --- Basic sanity checks ---
        if U.shape[0] != U.shape[1]:
            raise ValueError("U must be square")

        n = U.shape[0]

        # Optional: unitarity check
        if not np.allclose(U.conj().T @ U, np.eye(n), atol=atol):
            raise ValueError("Matrix is not unitary within tolerance")

        # --- Build adjacency graph of coupled basis states ---
        # Vertex: basis index k
        # Edge (i,j): if U[i,j] or U[j,i] has magnitude > atol and i != j
        adj = [[] for _ in range(n)]
        for i in range(n):
            for j in range(n):
                if i == j:
                    continue
                if np.abs(U[i, j]) > atol or np.abs(U[j, i]) > atol:
                    adj[i].append(j)

        # --- Find connected components via DFS/BFS ---
        visited = [False] * n
        components: List[List[int]] = []

        for v in range(n):
            if not visited[v]:
                stack = [v]
                comp = []
                visited[v] = True
                while stack:
                    x = stack.pop()
                    comp.append(x)
                    for y in adj[x]:
                        if not visited[y]:
                            visited[y] = True
                            stack.append(y)
                components.append(sorted(comp))

        # --- Classify components and extract 2-level blocks ---
        two_level_blocks: List[TwoLevelBlock] = []

        for comp in components:
            size = len(comp)

            if size == 1:
                # Only a phase on this basis state → ignore or log
                continue

            if size > 2:
                raise ValueError(
                    f"Found a {size}-dimensional mixing component {comp}; "
                    "this is not a 2-level unitary."
                )

            # size == 2 → two-level subspace
            i, j = comp  # sorted already

            # Extract the 2x2 submatrix on rows/cols (i,j)
            sub = np.zeros((2, 2), dtype=complex)
            sub[0, 0] = U[i, i]
            sub[0, 1] = U[i, j]
            sub[1, 0] = U[j, i]
            sub[1, 1] = U[j, j]

            # Optional: sanity check that "outside" rows/cols look like phases only
            for k in comp:
                row_nz = np.where(np.abs(U[k, :]) > atol)[0]
                col_nz = np.where(np.abs(U[:, k]) > atol)[0]
                # nonzeros must lie only in {i, j}
                if not set(row_nz).issubset({i, j}) or not set(col_nz).issubset({i, j}):
                    raise ValueError(
                        f"Basis state {k} has couplings outside {{i,j}}; "
                        "this is not a pure 2-level block."
                    )

            two_level_blocks.append(TwoLevelBlock(dim=n, i=i, j=j, submatrix=sub))

        return two_level_blocks
        
    def _apply_3qubit_unitary(self, qc, U, qargs):
        """
        Decompose and apply a 3-qubit energy-conserving unitary.
        
        Args:
            qc: QuantumCircuit to append gates to
            U: 8x8 unitary matrix
            qargs: [q0, q1, q2] physical qubit indices
        """
        q0, q1, q2 = qargs
        n = 8
        
        # Check energy conservation
        if not EnergyConservationDecompositionPass.is_energy_conserving(U):
            raise ValueError("Unitary is not energy-conserving")
        
        # Group basis states by Hamming weight
        hw_to_indices = {}
        for i in range(n):
            hw = bin(i).count('1')
            if hw not in hw_to_indices:
                hw_to_indices[hw] = []
            hw_to_indices[hw].append(i)
        
        # Extract and decompose each block
        print(f"\nDecomposing 3-qubit EC unitary:")
        
        ret_phases = []
        for hw in sorted(hw_to_indices.keys()):
            indices = hw_to_indices[hw]
            block_size = len(indices)
            
            # Extract block
            block = np.zeros((block_size, block_size), dtype=complex)
            for i, idx_i in enumerate(indices):
                for j, idx_j in enumerate(indices):
                    block[i, j] = U[idx_i, idx_j]
            
            print(f"\nHW={hw} block ({block_size}x{block_size}):")
            
            if block_size == 1:
                # Just a phase - can be handled with global phase or Rz gates
                phase = np.angle(block[0, 0])
                print(f"  Phase: {phase:.4f} rad (handled separately)")
                ret_phases.append(phase)
            else:
                # Reck decomposition
                tlus, D = reck_decomposition(block)
                print(f"  {len(tlus)} 2-level unitaries")
                
                # Apply each 2-level unitary (in reverse order for U = prod of U_k†)
                for tlu in reversed(tlus):
                    tlu_dag = tlu.dagger()
                    self._apply_2level_3qubit_unitary(qc, tlu_dag, qargs, indices)
                
                # Diagonal phases from D (can be absorbed into relative phases)
                diag_phases = [np.angle(D[i, i]) for i in range(block_size)]
                print(f"  Diagonal phases: {[f'{p:.3f}' for p in diag_phases]}")
                phase = diag_phases[0]  # representative phase
                ret_phases.append(phase)
        return ret_phases

   
   
    def _general_U_decomposition(self, qc, U, qargs):
        """
        Decompose and apply a general energy-conserving unitary U on n qubits.

        Args:
            qc: QuantumCircuit to append gates to
            U:  unitary matrix (2^n x 2^n)
            qargs: [q0, q1, q2, ....] physical qubit indices
        """
        n_qubits = len(qargs)
        dim = 2**n_qubits

        # Sanity checks
        if U.shape != (dim, dim):
            raise ValueError(
                f"Unitary shape {U.shape} incompatible with {n_qubits} qubits."
            )

        if not EnergyConservationDecompositionPass.is_energy_conserving(U):
            raise ValueError("Unitary is not energy-conserving")

        # Group basis states by Hamming weight
        hw_to_indices: dict[int, list[int]] = {}
        for i in range(dim):
            hw = bin(i).count("1")
            hw_to_indices.setdefault(hw, []).append(i)

        print(f"\nDecomposing {n_qubits}-qubit EC unitary:")

        # Keep track of per-HW phases (if you later want to fix relative phases)
        hw_phases: dict[int, list[float]] = {}
        
        blocks = EnergyConservationDecompositionPass.get_hamming_weight_blocks(U)

        for hw in sorted(hw_to_indices.keys()):
            indices = hw_to_indices[hw]
            block_size = len(indices)

            # Extract the block for this Hamming weight
            block = blocks[hw]
            print(f"\n  HW = {hw}, block size = {block.shape[0]}x{block.shape[1]}:")

            # Trivial 1x1 block: just a phase
            if block_size == 1:
                phase = np.angle(block[0, 0])
                print(f"    (1x1) phase = {phase:.4f} rad")
                hw_phases[hw] = [phase]
                continue
            
            # find 2-level blocks
            two_level_blocks = self.extract_two_level_blocks(block, atol=1e-10)
            print(f"    Found {len(two_level_blocks)} two-level blocks.")
            
            # Apply each 2-level block
            for tlb in two_level_blocks:
                tlb_dag = TwoLevelUnitary(tlb.dim, tlb.i, tlb.j, tlb.submatrix).dagger()
                self._apply_2level_nqubit_unitary(qc, tlb_dag, qargs, indices)
            
            

            

        # You can return hw_phases if you want to feed them into
        # a global relative-phase-fixing routine later.
        return hw_phases

        
        # TODO Then find basis elements with equal hamming weight and more than 2 hamming distance,
        # get the bit sqeuence from start to end by using iSWAP gates and apply the controlled exp_i_alpha_R and controlled exp_i_theta_L functions accordingly. 
        

        def _apply_2qubit_nqubit_unitary(self, qc, U, qargs):
            """
            Decompose and apply a 2-qubit 2-level unitary embedded in n-qubits.

            Args:
                qc: QuantumCircuit to append gates to
                U: 2^n x 2^n unitary matrix (2-level in computational basis)
                qargs: list of n physical qubit indices
            """
            n = len(qargs)

            # 1) Check that U is a 2-level unitary and locate the two basis states
            local_i, local_j = self._get_nontrivial_rows_cols(U)

            # 2) Extract the 2x2 submatrix for the nontrivial subspace
            submatrix = np.array([
                [U[local_i, local_i], U[local_i, local_j]],
                [U[local_j, local_i], U[local_j, local_j]]
            ], dtype=complex)

            # 3) Determine control bits and target bits from the two basis indices
            bits_i = bin(local_i)[2:].zfill(n)
            bits_j = bin(local_j)[2:].zfill(n)
            print(f"  Applying TLU: |{bits_i}⟩ ↔ |{bits_j}⟩ on local indices {local_i}, {local_j}")

            control_bits = []
            control_vals = []
            target_bits = []
            for bit in range(n):
                # Be careful with bit ordering; if you prefer MSB=qubit0, adjust accordingly
                bit_i = (local_i >> bit) & 1
                bit_j = (local_j >> bit) & 1
                if bit_i == bit_j:
                    control_bits.append(bit)
                    control_vals.append(bit_i)
                else:
                    target_bits.append(bit)

            if len(target_bits) != 2:
                raise ValueError("2-level unitary does not differ in exactly 2 bit positions.")

            print(f"    Controls: {list(zip(control_bits, control_vals))}, Targets: {target_bits}")

            # 4) Use recursive decomposition to break Λ_c(submatrix) into
            #    single-controlled 2-level unitaries.
            controlled_blocks = self._decompose_ncontrolled_2level_unitary(
                submatrix, control_bits, target_bits
            )

            # 5) Implement each ControlledTwoLevel using XYX and your gadgets
            for block in controlled_blocks:
                # SU(2) → XYX angles
                theta, phi, lam, global_phase = OneQubitEulerDecomposer('XYX') \
                    .angles_and_phase(Operator(block.U))

                if len(block.control_bits) != 1:
                    raise RuntimeError(
                        "Final decomposition should only contain single-controlled gates."
                    )

                ctrl_bit = block.control_bits[0]
                t0_bit, t1_bit = block.target_bits

                control = qargs[ctrl_bit]
                target1 = qargs[t0_bit]
                target2 = qargs[t1_bit]

                # What is the value of the control to condition on?
                # From the original control_vals list:
                ctrl_val = control_vals[control_bits.index(ctrl_bit)]

                print(
                    f"    Implementing single-controlled SU(2) on targets {block.target_bits} "
                    f"with control bit {ctrl_bit}={ctrl_val}, angles XYX:"
                    f" θ={theta:.3f}, φ={phi:.3f}, λ={lam:.3f}"
                )

                # Apply U = Rx(θ) Ry(φ) Rx(λ) via your gadgets:
                self._controlled_exp_i_alpha_R(lam, ctrl_val, qc, target1, target2, control)
                self._controlled_exp_i_theta_L(phi, ctrl_val, qc, target1, target2, control)
                self._controlled_exp_i_alpha_R(theta, ctrl_val, qc, target1, target2, control)

        def _apply_2qubit_nqubit_unitary(self, qc, U, qargs):
            """
            Decompose and apply a 2-qubit 2-level unitary embedded in n-qubits.

            Args:
                qc: QuantumCircuit to append gates to
                U: 2^n x 2^n unitary matrix (2-level in computational basis)
                qargs: list of n physical qubit indices
            """
            n = len(qargs)

            # 1) Check that U is a 2-level unitary and locate the two basis states
            local_i, local_j = self._get_nontrivial_rows_cols(U)

            # 2) Extract the 2x2 submatrix for the nontrivial subspace
            submatrix = np.array([
                [U[local_i, local_i], U[local_i, local_j]],
                [U[local_j, local_i], U[local_j, local_j]]
            ], dtype=complex)

            # 3) Determine control bits and target bits from the two basis indices
            bits_i = bin(local_i)[2:].zfill(n)
            bits_j = bin(local_j)[2:].zfill(n)
            print(f"  Applying TLU: |{bits_i}⟩ ↔ |{bits_j}⟩ on local indices {local_i}, {local_j}")

            control_bits = []
            control_vals = []
            target_bits = []
            for bit in range(n):
                # Be careful with bit ordering; if you prefer MSB=qubit0, adjust accordingly
                bit_i = (local_i >> bit) & 1
                bit_j = (local_j >> bit) & 1
                if bit_i == bit_j:
                    control_bits.append(bit)
                    control_vals.append(bit_i)
                else:
                    target_bits.append(bit)

            if len(target_bits) != 2:
                raise ValueError("2-level unitary does not differ in exactly 2 bit positions.")

            print(f"    Controls: {list(zip(control_bits, control_vals))}, Targets: {target_bits}")

            # 4) Use recursive decomposition to break Λ_c(submatrix) into
            #    single-controlled 2-level unitaries.
            controlled_blocks = self._decompose_ncontrolled_2level_unitary(
                submatrix, control_bits, target_bits
            )

            # 5) Implement each ControlledTwoLevel using XYX and your gadgets
            for block in controlled_blocks:
                # SU(2) → XYX angles
                theta, phi, lam, global_phase = OneQubitEulerDecomposer('XYX') \
                    .angles_and_phase(Operator(block.U))

                if len(block.control_bits) != 1:
                    raise RuntimeError(
                        "Final decomposition should only contain single-controlled gates."
                    )

                ctrl_bit = block.control_bits[0]
                t0_bit, t1_bit = block.target_bits

                control = qargs[ctrl_bit]
                target1 = qargs[t0_bit]
                target2 = qargs[t1_bit]

                # What is the value of the control to condition on?
                # From the original control_vals list:
                ctrl_val = control_vals[control_bits.index(ctrl_bit)]

                print(
                    f"    Implementing single-controlled SU(2) on targets {block.target_bits} "
                    f"with control bit {ctrl_bit}={ctrl_val}, angles XYX:"
                    f" θ={theta:.3f}, φ={phi:.3f}, λ={lam:.3f}"
                )

                # Apply U = Rx(θ) Ry(φ) Rx(λ) via your gadgets:
                self._controlled_exp_i_alpha_R(lam, ctrl_val, qc, target1, target2, control)
                self._controlled_exp_i_theta_L(phi, ctrl_val, qc, target1, target2, control)
                self._controlled_exp_i_alpha_R(theta, ctrl_val, qc, target1, target2, control)


    
    def _decompose_ncontrolled_2level_unitary(self, U: np.ndarray,
                                              control_bits: list[int],
                                              target_bits: list[int]) -> list[ControlledTwoLevel]:
        """
        Decompose a 2-level unitary with n control bits and 2 target bits.

        Args:
            U: 2x2 unitary matrix acting on the 2-level subspace.
            control_bits: list of control bit indices (k of them).
            target_bits: list of target bit indices (length 2).

        Returns:
            A list of ControlledTwoLevel objects. Each has some subset of controls;
            the recursion ends with k=1 (single-controlled 2-level).
        """
        k = len(control_bits)

        # --- Base case: single control bit ---
        if k == 1:
            # Just one multi-controlled gate Λ_c(U) → a single ControlledTwoLevel
            return [ControlledTwoLevel(U=U, control_bits=control_bits.copy(),
                                       target_bits=target_bits.copy())]

        # --- Recursive case: k ≥ 2 ---
        # Factor out global phase so that U_su2 ∈ SU(2)
        detU = np.linalg.det(U)
        U_su2 = U / np.sqrt(detU)

        # Find A(1), B(1) ∈ SU(2) s.t. A B A† B† = U_su2
        A1, B1 = self._su2_AB_from_U(U_su2)

        # Split the control bits into two halves: c1, c2
        mid = k // 2  # floor(k/2)
        c1 = control_bits[:mid]
        c2 = control_bits[mid:]

        # Recursive decomposition according to Eq. (64):
        #   Λ_c(U) = Λ_{c1}(A) Λ_{c2}(B) Λ_{c1}(A†) Λ_{c2}(B†)
        gates = []
        gates += self._decompose_ncontrolled_2level_unitary(A1, c1, target_bits)
        gates += self._decompose_ncontrolled_2level_unitary(B1, c2, target_bits)
        gates += self._decompose_ncontrolled_2level_unitary(A1.conj().T, c1, target_bits)
        gates += self._decompose_ncontrolled_2level_unitary(B1.conj().T, c2, target_bits)

        # Note: Any global phase from det(U) has been ignored at this stage.
        # You can account for it later if necessary via single-qubit Rz's.
        return gates
    
    def run(self, dag: DAGCircuit) -> DAGCircuit:
        """Run the pass on the given DAGCircuit.

        Args:
            dag (DAGCircuit): The input DAGCircuit to be transformed.

        Returns:
            DAGCircuit: The transformed DAGCircuit with decomposed gates.
        """
        circuit = dag_to_circuit(dag)
        
        anc = QuantumRegister(1, 'anc')
        qregs = dag.qregs
        cregs = dag.cregs
        
        new_circuit = QuantumCircuit(*qregs.values(), anc, *cregs.values())
        for instr, qargs, cargs in circuit.data:
            if instr.name == 'cz':
                q0 = qargs[0]._index
                q1 = qargs[1]._index
                self._apply_cz_decomposition(new_circuit, q0, q1, dag.num_qubits())
            
            elif instr.name == 'swap':
                q0 = qargs[0]._index
                q1 = qargs[1]._index
                self._apply_swap_decomposition(new_circuit, q0, q1, dag.num_qubits())
                
            elif instr.num_qubits == 2:
                q0 = qargs[0]._index
                q1 = qargs[1]._index
                phases = self._apply_2qubit_decomposition(new_circuit, instr, [q._index for q in qargs])
                self._fix_relative_phases(new_circuit, [q._index for q in qargs], phases)
            
            elif instr.num_qubits == 3:
                q0 = qargs[0]._index
                q1 = qargs[1]._index
                q2 = qargs[2]._index
                U = Operator(instr).data
                phases = self._apply_3qubit_unitary(new_circuit, U, [q._index for q in qargs])
                self._fix_relative_phases(new_circuit, [q._index for q in qargs], phases)
            
            else:
                new_circuit.append(instr, qargs, cargs)
        
        new_dag = circuit_to_dag(new_circuit)
        return new_dag


In [3]:
from qiskit.circuit.library import CZGate
from qiskit.quantum_info import Operator
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.converters import circuit_to_dag, dag_to_circuit
from qiskit.visualization.dag_visualization import dag_drawer
from qiskit.transpiler import PassManager
from scipy.linalg import expm
import numpy as np

### Explicitly Implemented
-  CZ
-  SWAP
-  2-level 3Qubit
-  3Qubit
-  

In [4]:
# Test the CZ decomposition
def test_apply_cz_decomposition():
    qc = QuantumCircuit(3)
    ancilla_index = 2  # Using qubit 2 as ancilla
    pass_instance = EnergyConservationDecompositionPass(target=None)
    pass_instance._apply_cz_decomposition(qc, 0, 1, ancilla_index)
    
    U_impl = Operator(qc).data
    
    effective_U = get_effective_unitary(qc, [ancilla_index], ancilla_state=0)
    
    
    # Ideal CZ gate on qubits 0 and 1
    qc_ideal = QuantumCircuit(3)
    qc_ideal.append(CZGate(), [0, 1])
    U_ideal = Operator(qc_ideal).data
    effective_U_ideal = get_effective_unitary(qc_ideal, [ancilla_index], ancilla_state=0)
    
    # print effective unitaries
    with np.printoptions(precision=3, suppress=True):
        print("Effective implemented unitary:\n", effective_U)
        print("Effective ideal unitary:\n", effective_U_ideal)
    
    print("||effective_U - effective_U_ideal||_max =", np.max(np.abs(effective_U - effective_U_ideal)))

test_apply_cz_decomposition()

Effective implemented unitary:
 [[ 1.+0.j  0.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j  1.+0.j -0.-0.j  0.+0.j]
 [ 0.+0.j  0.-0.j  1.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  0.+0.j -1.+0.j]]
Effective ideal unitary:
 [[ 1.+0.j  0.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j  1.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  1.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  0.+0.j -1.+0.j]]
||effective_U - effective_U_ideal||_max = 2.220446049250313e-16


In [5]:
# Test the SWAP decomposition
def test_apply_swap_decomposition():
    qc = QuantumCircuit(3)
    ancilla_index = 2  # Using qubit 2 as ancilla
    pass_instance = EnergyConservationDecompositionPass(target=None)
    pass_instance._apply_swap_decomposition(qc, 0, 1, ancilla_index)
    
    U_impl = Operator(qc).data
    
    effective_U = get_effective_unitary(qc, [ancilla_index], ancilla_state=0)
    
    
    # Ideal SWAP gate on qubits 0 and 1
    qc_ideal = QuantumCircuit(3)
    qc_ideal.swap(0, 1)
    U_ideal = Operator(qc_ideal).data
    effective_U_ideal = get_effective_unitary(qc_ideal, [ancilla_index], ancilla_state=0)
    
    # print effective unitaries
    with np.printoptions(precision=3, suppress=True):
        print("Effective implemented unitary:\n", effective_U)
        print("Effective ideal unitary:\n", effective_U_ideal)
    
    print("||effective_U - effective_U_ideal||_max =", np.max(np.abs(effective_U - effective_U_ideal)))
test_apply_swap_decomposition()

Effective implemented unitary:
 [[ 1.+0.j  0.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j  0.-0.j  1.+0.j  0.+0.j]
 [ 0.+0.j  1.+0.j -0.-0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  0.+0.j  1.-0.j]]
Effective ideal unitary:
 [[1.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 1.+0.j 0.+0.j]
 [0.+0.j 1.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 1.+0.j]]
||effective_U - effective_U_ideal||_max = 2.7755575615628914e-16


In [6]:
# Test the _exp_i_alpha_R function
def test_exp_i_alpha_R_identity_and_group():
    obj = EnergyConservationDecompositionPass(target=None)

    # --- Identity at theta = 0 ---
    qc_id = QuantumCircuit(2)
    obj._exp_i_alpha_R(0.0, qc_id, 0, 1)
    U_id = Operator(qc_id).data
    assert np.allclose(U_id, np.eye(4, dtype=complex), atol=1e-8), \
        "_exp_i_alpha_R(0) is not identity"

    # --- Group property: U(theta1+theta2) = U(theta2) U(theta1) ---
    theta1 = 0.37
    theta2 = -0.81

    # Direct: U(theta1+theta2)
    qc_direct = QuantumCircuit(2)
    obj._exp_i_alpha_R(theta1 + theta2, qc_direct, 0, 1)
    U_direct = Operator(qc_direct).data

    # Composition: U(theta2) · U(theta1)
    qc_comp = QuantumCircuit(2)
    obj._exp_i_alpha_R(theta1, qc_comp, 0, 1)
    obj._exp_i_alpha_R(theta2, qc_comp, 0, 1)
    U_comp = Operator(qc_comp).data

    assert _unitaries_close_up_to_phase(U_direct, U_comp), \
        "Group property failed for _exp_i_alpha_R"
    
    print("test_exp_i_alpha_R_identity_and_group passed.")

test_exp_i_alpha_R_identity_and_group()

test_exp_i_alpha_R_identity_and_group passed.


In [7]:
# Test the _controlled_exp_i_alpha_R function
def _ideal_controlled_R(theta, b, obj):
    """
    Build the ideal 3-qubit unitary for controlled exp_i_alpha_R(theta)
    with control on qubit 2, targets on (0,1), conditioned on value b.
    """
    # 2-qubit R unitary
    qc_R = QuantumCircuit(2)
    obj._exp_i_alpha_R(theta, qc_R, 0, 1)
    U_R = Operator(qc_R).data  # 4x4

    # Full 3-qubit ideal: block-diagonal
    # basis: |000>=0..|011>=3 (control=0), |100>=4..|111>=7 (control=1)
    U_ideal = np.eye(8, dtype=complex)

    if b == 0:
        # apply U_R when control qubit = 0 → upper-left 4x4 block
        U_ideal[0:4, 0:4] = U_R
        # leave lower block identity
    else:
        # apply U_R when control qubit = 1 → lower-right 4x4 block
        U_ideal[4:8, 4:8] = U_R
        # leave upper block identity

    return U_ideal

def test_controlled_exp_i_alpha_R_identity():
    obj = EnergyConservationDecompositionPass(target=None)
    qc = QuantumCircuit(3)
    obj._controlled_exp_i_alpha_R(0.0, b=0, qc=qc, q0=0, q1=1, control=2)
    U = Operator(qc).data
    assert np.allclose(U, np.eye(8, dtype=complex), atol=1e-8), \
        "_controlled_exp_i_alpha_R(0) is not identity"


def test_controlled_exp_i_alpha_R_against_ideal():
    obj = EnergyConservationDecompositionPass(target=None)
    theta = 0.53

    for b in [0, 1]:
        # implemented
        qc_impl = QuantumCircuit(3)
        obj._controlled_exp_i_alpha_R(theta, b, qc_impl, 0, 1, 2)
        U_impl = Operator(qc_impl).data

        # ideal
        U_ideal = _ideal_controlled_R(theta, b, obj)

        assert _unitaries_close_up_to_phase(U_impl, U_ideal), \
            f"_controlled_exp_i_alpha_R mismatch for b={b}"

def test_controlled_exp_i_alpha_R_conditioning():
    obj = EnergyConservationDecompositionPass(target=None)
    theta = 0.77
    b = 1  # test for one value; you can also loop over both

    qc_impl = QuantumCircuit(3)
    obj._controlled_exp_i_alpha_R(theta, b, qc_impl, 0, 1, 2)
    U_impl = Operator(qc_impl).data
    
    # with np.printoptions(precision=3, suppress=True, linewidth=120):
    #     print("Implemented U:\n", U_impl)
    #     # Ideal U:
    #     U_ideal = _ideal_controlled_R(theta, b, obj)
    #     print("Ideal U:\n", U_ideal)
    # states where control bit (q2) != b should be unchanged (up to phase)
    for idx in range(8):
        bits = format(idx, "03b")
        control_val = int(bits[0])  # q2 is rightmost bit
        if control_val != b:
            e = np.zeros(8, dtype=complex)
            e[idx] = 1.0
            v = U_impl @ e
            # v should be proportional to e
            nonzero = np.where(np.abs(v) > 1e-8)[0]
            assert len(nonzero) == 1 and nonzero[0] == idx, \
                "Control off-state leaked to other basis vectors"
    print("test_controlled_exp_i_alpha_R passed.")

# Run the tests
test_controlled_exp_i_alpha_R_identity()
test_controlled_exp_i_alpha_R_against_ideal()
test_controlled_exp_i_alpha_R_conditioning()

test_controlled_exp_i_alpha_R passed.


In [8]:
# Test the _controlled_exp_i_theta_L function
def test_controlled_exp_i_theta_L_identity_and_group():
    obj = EnergyConservationDecompositionPass(target=None)
    b = 1

    # identity check
    qc_id = QuantumCircuit(3)
    obj._controlled_exp_i_theta_L(0.0, b, qc_id, 0, 1, 2)
    U_id = Operator(qc_id).data
    assert np.allclose(U_id, np.eye(8, dtype=complex), atol=1e-8), \
        "_controlled_exp_i_theta_L(0) is not identity"

    # group property
    theta1 = 0.41
    theta2 = -0.66

    qc_direct = QuantumCircuit(3)
    obj._controlled_exp_i_theta_L(theta1 + theta2, b, qc_direct, 0, 1, 2)
    U_direct = Operator(qc_direct).data

    qc_comp = QuantumCircuit(3)
    obj._controlled_exp_i_theta_L(theta1, b, qc_comp, 0, 1, 2)
    obj._controlled_exp_i_theta_L(theta2, b, qc_comp, 0, 1, 2)
    U_comp = Operator(qc_comp).data

    assert _unitaries_close_up_to_phase(U_direct, U_comp), \
        "Group property failed for _controlled_exp_i_theta_L"

def test_controlled_exp_i_theta_L_conditioning():
    obj = EnergyConservationDecompositionPass(target=None)
    theta = 0.6
    b = 0

    qc_impl = QuantumCircuit(3)
    obj._controlled_exp_i_theta_L(theta, b, qc_impl, 0, 1, 2)
    U_impl = Operator(qc_impl).data

    # states where control bit (q2) != b should be eigenvectors with no leakage
    for idx in range(8):
        bits = format(idx, "03b")
        control_val = int(bits[0])  # q2 is rightmost
        if control_val != b:
            e = np.zeros(8, dtype=complex)
            e[idx] = 1.0
            v = U_impl @ e
            nonzero = np.where(np.abs(v) > 1e-8)[0]
            assert len(nonzero) == 1 and nonzero[0] == idx, \
                "_controlled_exp_i_theta_L leaked when control off"

# Run the tests
test_controlled_exp_i_theta_L_identity_and_group()
test_controlled_exp_i_theta_L_conditioning()

In [9]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator

# --------------------------------------------------------------------
# 1. Build a 3-qubit two-level unitary (8x8)
#    Non-trivial only on |001> and |010>
# --------------------------------------------------------------------
def make_3q_two_level_unitary():
    """
    Construct an 8x8 unitary that is identity except for a 2x2 SU(2) block
    mixing |001> (index 1) and |010> (index 2).

    Returns:
        U: 8x8 np.ndarray complex
    """
    # Generic SU(2) block
    theta = np.pi /8
    phi = np.pi /8

    c = np.cos(theta / 2.0)
    s = np.sin(theta / 2.0)

    # Example SU(2) on span{|001>, |010>}
    sub = np.array(
        [
            [c, -np.exp(1j * phi) * s],
            [np.exp(-1j * phi) * s, c],
        ],
        dtype=complex,
    )

    # Full 3-qubit unitary: identity except this 2x2 block
    U = np.eye(8, dtype=complex)

    # basis: |000>=0, |001>=1, |010>=2, |011>=3,
    #        |100>=4, |101>=5, |110>=6, |111>=7
    i = 1  # |001>
    j = 2  # |010>

    U[i, i] = sub[0, 0]
    U[i, j] = sub[0, 1]
    U[j, i] = sub[1, 0]
    U[j, j] = sub[1, 1]
    
    # print the euler angles of the submatrix
    theta_x, phi_y, lam_x, global_phase = OneQubitEulerDecomposer('XYX').angles_and_phase(
        Operator(sub)
    )
    print(f"Euler angles (XYX) of submatrix: theta={theta_x}, phi={phi_y}, lambda={lam_x}, global_phase={global_phase}")

    return U


# --------------------------------------------------------------------
# 2. Test _apply_2level_3qubit_unitary with full 8x8 U
# --------------------------------------------------------------------
def test_apply_2level_3qubit_unitary():
    # build target unitary
    U_target = make_3q_two_level_unitary()

    # new circuit
    qc = QuantumCircuit(3)

    # instantiate your pass
    # from your_module import EnergyConservationDecompositionPass
    pass_obj = EnergyConservationDecompositionPass(target=None)

    # physical qubit mapping
    qargs = [0, 1, 2]

    # hw_indices isn't actually used in your new implementation,
    # but we'll pass the natural mapping 0..7 for completeness.
    hw_indices = list(range(8))

    # apply your 2-level decomposition
    pass_obj._apply_2level_3qubit_unitary(qc, U_target, qargs)

    # implemented unitary
    U_impl = Operator(qc).data
    
    with np.printoptions(precision=3, suppress=True, linewidth=10000):
        print("Implemented unitary U_impl:\n", U_impl)
        print("Target unitary U_target:\n", U_target)
    
    # get the submatrix corresponding to |001> and |010>
    sub_impl = np.array(
        [
            [U_impl[1, 1], U_impl[1, 2]],
            [U_impl[2, 1], U_impl[2, 2]],
        ],
        dtype=complex,
    )
    sub_target = np.array(
        [
            [U_target[1, 1], U_target[1, 2]],
            [U_target[2, 1], U_target[2, 2]],
        ],
        dtype=complex,
    )
    xyx_decomposer = OneQubitEulerDecomposer('XYX')
    with np.printoptions(precision=3, suppress=True):
        print("Implemented submatrix:\n", sub_impl)
        print("Target submatrix:\n", sub_target)
        theta_impl, phi_impl, lam_impl, phase_impl = xyx_decomposer.angles_and_phase(
            Operator(sub_impl)
        )
        theta_target, phi_target, lam_target, phase_target = xyx_decomposer.angles_and_phase(
            Operator(sub_target)
        )
        # print(f"Implemented submatrix angles: theta={theta_impl}, phi={phi_impl}, lambda={lam_impl}, phase={phase_impl}")
        # print(f"Target submatrix angles: theta={theta_target}, phi={phi_target}, lambda={lam_target}, phase={phase_target}")
        
        

    # compare
    diff = U_impl - U_target
    max_err = np.max(np.abs(diff))
    print("Max entry-wise error ||U_impl - U_target||_max =", max_err)

    # If you want a hard test:
    # assert max_err < 1e-8, "Two-level 3-qubit decomposition failed tolerance!"
    
    # test upto global phase
    assert _unitaries_close_up_to_phase(sub_impl, sub_target), "Two-level 3-qubit decomposition failed up to global phase!"



test_apply_2level_3qubit_unitary()


Euler angles (XYX) of submatrix: theta=0.3624607931208141, phi=0.07597395426252618, lambda=0.07597395426252662, global_phase=-2.220446049250313e-16


NameError: name 'TwoLevelUnitary' is not defined

In [25]:
from implementation import EnergyConservingDecompositionPass
from qiskit import QuantumCircuit, transpile
from qiskit.transpiler import PassManager
import numpy as np
from qiskit.circuit.library import UnitaryGate, CCZGate, CSwapGate
from qiskit.quantum_info import Operator

def exp_i_theta_involution(A: np.ndarray, theta: float) -> np.ndarray:
    """
    Compute exp(i θ A) for a matrix A with A^2 = I.
    Uses: exp(i θ A) = cos θ I + i sin θ A
    """
    dim = A.shape[0]
    I = np.eye(dim, dtype=complex)
    return np.cos(theta) * I + 1j * np.sin(theta) * A


def add_exp_i_theta_swap(qc: QuantumCircuit, theta: float, q0: int, q1: int):
    """Append e^{i θ SWAP} acting on qubits (q0, q1)."""
    swap = np.array(
        [
            [1, 0, 0, 0],
            [0, 0, 1, 0],
            [0, 1, 0, 0],
            [0, 0, 0, 1],
        ],
        dtype=complex,
    )
    U = exp_i_theta_involution(swap, theta)
    # gate = UnitaryGate(U, label="exp(iθ SWAP)")
    # qc.append(gate, [q0, q1])
    return U

def make_2qubit_ec_unitary():
    alpha = np.pi / 2
    beta = np.pi / 2
    theta = np.pi / 2

    # SU(2) rotation in {|01>, |10>}
    # here take R_y(theta) = [[cos(θ/2), -sin(θ/2)],
    #                         [sin(θ/2),  cos(θ/2)]]
    c = np.cos(theta/2)
    s = np.sin(theta/2)
    block_1 = np.array([[c, -s],
                        [s,  c]], dtype=complex)

    U = np.zeros((4,4), dtype=complex)
    
    theta, pi, lam, global_phase = OneQubitEulerDecomposer('ZYZ').angles_and_phase(
        Operator(block_1)
    )
    print(f"Euler angles (ZYZ) of 2-qubit block: theta={theta}, phi={pi}, lambda={lam}, global_phase={global_phase}")

    # basis: |00>, |01>, |10>, |11>
    U[0,0] = np.exp(1j*alpha)   # HW = 0
    U[1:3, 1:3] = block_1       # HW = 1
    U[3,3] = np.exp(1j*beta)    # HW = 2

    return U

def test_apply_2qubit_decomposition():
    # U = make_2qubit_ec_unitary()
    U = add_exp_i_theta_swap(QuantumCircuit(2), theta=np.pi/3, q0=0, q1=1)
    U = U * np.exp(-1j * np.angle(U[0, 0]))  # global phase correction
    instr = Operator(U).to_instruction()
    
    with np.printoptions(precision=3, suppress=True):
        print("Target 2-qubit EC unitary U:\n", U)

    qc = QuantumCircuit(3)
    pass_obj = EnergyConservationDecompositionPass(target=None)
    phases = pass_obj._apply_2qubit_decomposition(qc, instr, [0,1, 2])
    # pass_obj._fix_relative_phases(qc, [0, 1], 2, phases)
    
    # effective_U = get_effective_unitary(qc, [2], ancilla_state=0)
    effective_U = Operator(qc).data
    # effective_U = effective_U * np.exp(-1j * np.angle(effective_U[0, 0]))  # global phase correction
    with np.printoptions(precision=3, suppress=True, linewidth=1000):
        print("Effective implemented unitary:\n", effective_U)
        print("Target unitary:\n", U)
        print("||effective_U - U||_max =", np.max(np.abs(effective_U - U)))

    U_impl = Operator(qc).data
    print("||U_impl - U||_max =", np.max(np.abs(effective_U - U)))
    print("Returned phases:", phases)

test_apply_2qubit_decomposition()

Target 2-qubit EC unitary U:
 [[1.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j   ]
 [0.  +0.j    0.25-0.433j 0.75+0.433j 0.  +0.j   ]
 [0.  +0.j    0.75+0.433j 0.25-0.433j 0.  +0.j   ]
 [0.  +0.j    0.  +0.j    0.  +0.j    1.  +0.j   ]]
2-qubit EC block (HW=1): [[0.25-0.4330127j 0.75+0.4330127j]
 [0.75+0.4330127j 0.25-0.4330127j]]
2-qubit EC block angles: θ=2.094, φ=1.571, λ=-1.571, global_phase=-1.047
Effective implemented unitary:
 [[ 1. -0.j     0. +0.j     0. +0.j     0. +0.j     0. +0.j     0. +0.j     0. +0.j     0. +0.j   ]
 [ 0. +0.j     0.5+0.j     0. +0.866j  0. +0.j     0. +0.j     0. +0.j     0. +0.j     0. +0.j   ]
 [ 0. +0.j    -0. +0.866j  0.5-0.j     0. +0.j     0. +0.j     0. +0.j     0. +0.j     0. +0.j   ]
 [ 0. +0.j     0. +0.j     0. +0.j     1. +0.j     0. +0.j     0. +0.j     0. +0.j     0. +0.j   ]
 [ 0. +0.j     0. +0.j     0. +0.j     0. +0.j     1. -0.j     0. +0.j     0. +0.j     0. +0.j   ]
 [ 0. +0.j     0. +0.j     0. +0.j     0. +0.j     0. +0.j     0.5+0.

ValueError: operands could not be broadcast together with shapes (8,8) (4,4) 

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit.library import SwapGate
from qiskit.quantum_info import Operator

# ---------------------------------------------------------
# Helper: compare unitaries up to global phase
# ---------------------------------------------------------
def unitaries_close_up_to_phase(U, V, atol=1e-8):
    """Check if U and V are equal up to a global phase."""
    # <V|U> as a complex number
    uv = np.vdot(V.flatten(), U.flatten())
    if np.abs(uv) < 1e-12:
        # fallback: plain allclose if inner product is tiny
        return np.allclose(U, V, atol=atol)
    phase = np.angle(uv)
    U_phase = np.exp(-1j * phase) * U
    return np.allclose(U_phase, V, atol=atol)


# ---------------------------------------------------------
# Build exp(i θ SWAP_01 ⊗ SWAP_23)
# ---------------------------------------------------------
def make_exp_i_theta_swap12_swap34(theta):
    """
    Construct U = exp(i θ (SWAP_01 · SWAP_23)) for 4 qubits.
    Uses S^2 = I => exp(i θ S) = cos θ I + i sin θ S.
    """
    # 4-qubit SWAP_01 followed by SWAP_23
    qc_swap = QuantumCircuit(4)
    qc_swap.append(SwapGate(), [0, 1])
    qc_swap.append(SwapGate(), [2, 3])
    S = Operator(qc_swap).data  # 16x16 involution

    dim = S.shape[0]
    I = np.eye(dim, dtype=complex)

    U = np.cos(theta) * I + 1j * np.sin(theta) * S
    return U


# ---------------------------------------------------------
# The test itself
# ---------------------------------------------------------
def test_decompose_exp_i_theta_swap12_swap34():
    # choose some nontrivial angle
    theta = np.pi / 2

    # 1) Ideal target unitary
    U_target = make_exp_i_theta_swap12_swap34(theta)
    print(f"SHape of U_target: {U_target.shape}")
    with np.printoptions(precision=3, suppress=True, linewidth=200):
        print("Target unitary U_target:\n", U_target.imag)


    qc_impl = QuantumCircuit(4)
    decomposer = EnergyConservationDecompositionPass(target=None)
    # qargs in order [0,1,2,3]
    hw_phases = decomposer._general_U_decomposition(qc_impl, U_target, [0, 1, 2, 3])

    U_impl = Operator(qc_impl).data

    # 3) Print both matrices (rounded) for inspection
    np.set_printoptions(precision=3, suppress=True)

    print("=== Expected U (exp(i θ SWAP_01 SWAP_23)) ===")
    print(U_target)
    print("\n=== Implemented U (from EC decomposer) ===")
    print(U_impl)

    # 4) Check they match up to global phase
    assert unitaries_close_up_to_phase(U_impl, U_target), \
        "exp(i θ SWAP_01 SWAP_23) decomposition failed!"

test_decompose_exp_i_theta_swap12_swap34()


SHape of U_target: (16, 16)
Target unitary U_target:
 [[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]]

Decomposing 4-qubit EC unitary:

  HW = 0, block size = 1x1:
    (1x1) phase = 1.5708 rad

  HW = 1, block size = 4x4:
    Reck:

NotImplementedError: 2-level unitary couples |0110> and |1001> with Hamming distance 4, but current implementation assumes distance = 2.